In [4]:
import sys, platform
import torch, numpy as np, pandas as pd
import sklearn, transformers, datasets

try:
    import torchcrf
    torchcrf_ver = getattr(torchcrf, "__version__", "unknown")
except Exception:
    torchcrf_ver = "not found"

try:
    import seqeval
    seqeval_ver = getattr(seqeval, "__version__", "unknown")
except Exception:
    seqeval_ver = "not found"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("CUDA:", torch.version.cuda)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("torchcrf:", torchcrf_ver)
print("seqeval:", seqeval_ver)
print("scikit-learn:", sklearn.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


Python: 3.12.12
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
GPU: NVIDIA A100-SXM4-40GB
CUDA: 12.6
torch: 2.9.0+cu126
transformers: 4.57.3
datasets: 4.0.0
torchcrf: 0.7.2
seqeval: unknown
scikit-learn: 1.6.1
numpy: 2.0.2
pandas: 2.2.2


# scibert

In [ ]:
# ===== کتابخانه‌ها =====
import os, json
import torch
import numpy as np
import pandas as pd
from collections import Counter
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint
from torchcrf import CRF
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import List, Dict, Any
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix


# ===== نام مدل =====
MODEL_NAME = "allenai/scibert_scivocab_cased"

torch.backends.cuda.matmul.allow_tf32 = True

if torch.cuda.is_available():
    torch.cuda.empty_cache()

def read_iob_file(file_path):
    sentences, labels = [], []
    tokens, tags = [], []

    with open(file_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                parts = line.split()
                tok, tag = parts[0], parts[-1]
                tokens.append(tok)
                tags.append(tag)

        if tokens:
            sentences.append(tokens)
            labels.append(tags)

    return sentences, labels


# ===== مسیر فایل‌ها =====
train_file = "/content/drive/MyDrive/poroje/data/merged_train_iob_fixed.txt"
test_file  = "/content/drive/MyDrive/poroje/data/merged_test_iob_fixed.txt"

train_sents, train_tags = read_iob_file(train_file)
test_sents, test_tags = read_iob_file(test_file)

unique_labels = sorted({
    t for seq in train_tags + test_tags for t in seq
})

combined_labels = ["O"] + [
    l for l in unique_labels if l != "O"
]

label2id = {l: i for i, l in enumerate(combined_labels)}
id2label = {i: l for l, i in label2id.items()}

flat_tags = [t for seq in train_tags for t in seq]

counts = Counter(flat_tags)
total = sum(counts.values())
num_lbls = len(combined_labels)

class_weights = {
    lbl: total / (num_lbls * counts.get(lbl, 1))
    for lbl in combined_labels
}

weights_tensor = torch.tensor(
    [class_weights[lbl] for lbl in combined_labels],
    dtype=torch.float
)

for cls in ["I-TASK", "I-DATASET", "B-TECHNOLOGY", "I-TECHNOLOGY"]:
    if cls in label2id:
        weights_tensor[label2id[cls]] *= 2.0

train_examples = [
    {"tokens": s, "ner_tags": [label2id[t] for t in ts]}
    for s, ts in zip(train_sents, train_tags)
]

eval_examples = [
    {"tokens": s, "ner_tags": [label2id[t] for t in ts]}
    for s, ts in zip(test_sents, test_tags)
]

datasets = DatasetDict({
    "train": Dataset.from_list(train_examples),
    "eval": Dataset.from_list(eval_examples),
})

print(datasets)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

MAX_LEN = 320


def tokenize_and_align_iob(ex):
    enc = tokenizer(
        ex["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    word_ids = enc.word_ids()
    _ = enc.pop("offset_mapping")

    labels_aln, token_mask = [], []
    prev = None

    for wi in word_ids:

        if wi is None:
            labels_aln.append(label2id["O"])
            token_mask.append(0)

        elif wi != prev:
            labels_aln.append(ex["ner_tags"][wi])
            token_mask.append(1)

        else:
            tag = combined_labels[ex["ner_tags"][wi]]

            if tag.startswith("B-"):
                tag = tag.replace("B-", "I-")

            labels_aln.append(label2id[tag])
            token_mask.append(1)

        prev = wi

    enc["labels"] = labels_aln
    enc["token_mask"] = token_mask

    return enc


tokenized = datasets.map(
    tokenize_and_align_iob,
    batched=False,
    remove_columns=["tokens", "ner_tags"]
)

print(tokenized)

@dataclass
class MyTokenCollator:
    tokenizer: Any
    label_pad_token_id: int

    def __call__(
        self,
        features: List[Dict[str, Any]]
    ) -> Dict[str, torch.Tensor]:

        labels = [f.pop("labels") for f in features]
        masks = [f.pop("token_mask") for f in features]

        batch = self.tokenizer.pad(
            features,
            return_tensors="pt"
        )

        max_len = batch["input_ids"].size(1)

        def pad_to(x, pad_val, L):
            return x + [pad_val] * (L - len(x))

        batch["labels"] = torch.tensor(
            [
                pad_to(l, self.label_pad_token_id, max_len)
                for l in labels
            ],
            dtype=torch.long
        )

        batch["token_mask"] = torch.tensor(
            [
                pad_to(m, 0, max_len)
                for m in masks
            ],
            dtype=torch.bool
        )

        return batch


data_collator = MyTokenCollator(
    tokenizer,
    label_pad_token_id=label2id["O"]
)


# =====مدل SciBERT + WordTransformer + Linear + CRF
#        + Boundary Head =====
class BertCRFForTokenClassification(nn.Module):

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        labels_vocab,
        feature_extraction: bool = False,
        class_weights: torch.Tensor = None,
        focal_alpha: float = 2.0,
        focal_gamma: float = 3.0,
        boundary_beta: float = 0.3
    ):

        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        if feature_extraction:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.dropout1 = nn.Dropout(0.3)
        self.dropout2 = nn.Dropout(0.3)

        self.word_transformer_layer = TransformerEncoderLayer(
            d_model=self.bert.config.hidden_size,
            nhead=8,
            dim_feedforward=self.bert.config.hidden_size * 4,
            batch_first=True
        )

        self.word_transformer = TransformerEncoder(
            self.word_transformer_layer,
            num_layers=1
        )

        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_labels
        )

        self.crf = CRF(
            num_labels,
            batch_first=True
        )

        
        self.boundary_classifier = nn.Linear(
            self.bert.config.hidden_size,
            2
        )

        self.class_weights = class_weights
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.boundary_beta = boundary_beta
        self.labels_vocab = labels_vocab


        is_B_list = [
            1 if lbl.startswith("B-") else 0
            for lbl in labels_vocab
        ]

        self.register_buffer(
            "is_B",
            torch.tensor(is_B_list, dtype=torch.long)
        )

        self._apply_bio_constraints()


    def _apply_bio_constraints(self):

        L = self.labels_vocab

        def typ(lbl):
            return lbl.split('-', 1)[1] if lbl != "O" else ""

        with torch.no_grad():

            for i, li in enumerate(L):

                if li.startswith("I-"):
                    self.crf.start_transitions[i] = -1e4

                for j, lj in enumerate(L):

                    if lj.startswith("I-"):

                        ok_prev = (
                            li == f"B-{typ(lj)}"
                        ) or (
                            li == f"I-{typ(lj)}"
                        )

                        if not ok_prev:
                            self.crf.transitions[i, j] = -1e4


    def forward(
        self,
        input_ids,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        token_mask=None
    ):
        x = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        ).last_hidden_state

        x = self.dropout1(x)

        valid_mask = (
            token_mask.bool()
            if token_mask is not None
            else attention_mask.bool()
        )

        x = self.word_transformer(
            x,
            src_key_padding_mask=~valid_mask
        )

        x = self.dropout2(x)

        emissions = self.classifier(x)

        boundary_logits = self.boundary_classifier(x)

        if labels is not None:

            b, s, n = emissions.size()

            em = emissions.reshape(-1, n)[
                valid_mask.reshape(-1)
            ]

            lb = labels.reshape(-1)[
                valid_mask.reshape(-1)
            ]

            ce = F.cross_entropy(
                em,
                lb,
                weight=(
                    self.class_weights.to(em.device)
                    if self.class_weights is not None
                    else None
                ),
                reduction="none"
            )

            pt = torch.exp(-ce)

            focal = (
                self.focal_alpha
                * (1 - pt) ** self.focal_gamma
                * ce
            ).mean()

            crf_loss = -self.crf(
                emissions,
                labels,
                mask=valid_mask,
                reduction="mean"
            )
            boundary_targets = self.is_B[labels]
            bl = boundary_logits.reshape(-1, 2)[
                valid_mask.reshape(-1)
            ]

            bt = boundary_targets.reshape(-1)[
                valid_mask.reshape(-1)
            ]

            boundary_loss = F.cross_entropy(
                bl,
                bt,
                reduction="mean"
            )
            total_loss = (
                crf_loss
                + focal
                + self.boundary_beta * boundary_loss
            )


            return {
                "loss": total_loss,
                "logits": emissions,
                "boundary_loss": boundary_loss.detach()
            }
        best_paths = self.crf.decode(
            emissions,
            mask=valid_mask
        )

        max_len = emissions.size(1)

        padded = [
            seq + [0] * (max_len - len(seq))
            for seq in best_paths
        ]

        return torch.tensor(
            padded,
            device=emissions.device
        )


# ===== Training =====
EPOCHS = 30

CHECKPOINT_DIR = "./temp"

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

model = BertCRFForTokenClassification(
    MODEL_NAME,
    len(combined_labels),
    labels_vocab=combined_labels,
    feature_extraction=False,
    class_weights=weights_tensor,
    focal_alpha=2.0,
    focal_gamma=3.0,
    boundary_beta=0.3
)


training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    learning_rate=1e-5,
    weight_decay=0.1,
    warmup_ratio=0.3,

    logging_dir=os.path.join(CHECKPOINT_DIR, "logs"),
    logging_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    load_best_model_at_end=False,

    bf16=True,
    tf32=True,
    optim="adamw_torch_fused",

    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    do_train=True,
    do_eval=True,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

last_ckpt = get_last_checkpoint(
    CHECKPOINT_DIR
)

trainer.train(
    resume_from_checkpoint=(
        last_ckpt
        if last_ckpt is not None
        else None
    )
)

eval_results = trainer.evaluate()

print("\n=== Eval (HF; loss-based) ===")
print(eval_results)

eval_loader = DataLoader(
    tokenized["eval"],
    batch_size=16,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=2
)


model.eval()

device = next(model.parameters()).device

all_true, all_pred = [], []


with torch.no_grad():

    for batch in eval_loader:

        labels = batch.pop("labels")
        tmask = batch["token_mask"]

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        decoded_ids = model(**batch).cpu()


        for true_ids, pred_ids, m in zip(
            labels,
            decoded_ids,
            tmask
        ):

            seq_true, seq_pred = [], []

            for t_id, p_id, mk in zip(
                true_ids,
                pred_ids,
                m
            ):

                if mk:
                    seq_true.append(
                        id2label[int(t_id)]
                    )

                    seq_pred.append(
                        id2label[int(p_id)]
                    )

            all_true.append(seq_true)
            all_pred.append(seq_pred)


# ===== Classification Report =====
print("\n--- Classification Report")

print(
    classification_report(
        all_true,
        all_pred,
        digits=4
    )
)

print(
    f"\nOverall Precision: {precision_score(all_true, all_pred):.4f}"
)

print(
    f"Overall Recall:    {recall_score(all_true, all_pred):.4f}"
)

print(
    f"Overall F1:        {f1_score(all_true, all_pred):.4f}"
)


# ===== ماتریس درهم‌ریختگی =====
flat_true = [
    lbl
    for seq in all_true
    for lbl in seq
]

flat_pred = [
    lbl
    for seq in all_pred
    for lbl in seq
]

cm = confusion_matrix(
    flat_true,
    flat_pred,
    labels=combined_labels
)

df_cm = pd.DataFrame(
    cm,
    index=combined_labels,
    columns=combined_labels
)

print("\n--- Confusion Matrix ---")
print(df_cm)

Cleaned sentences: 57263 14524
Num labels: 21
['O', 'B-DATASET', 'B-GENERIC', 'B-MATERIAL', 'B-METHOD', 'B-METRIC', 'B-MODEL', 'B-OTHERSCIENTIFICTERM', 'B-TASK', 'B-TECHNOLOGY', 'B-TOOLS', 'I-DATASET', 'I-GENERIC', 'I-MATERIAL', 'I-METHOD', 'I-METRIC', 'I-MODEL', 'I-OTHERSCIENTIFICTERM', 'I-TASK', 'I-TECHNOLOGY', 'I-TOOLS']
Train sentences: 57263  -> after OS: 60109


Map:   0%|          | 0/60109 [00:00<?, ? examples/s]

Map:   0%|          | 0/14524 [00:00<?, ? examples/s]

/tmp/ipython-input-3345610147.py:479: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
100,76.866400
200,51.778200
300,29.696700
400,21.899500
500,17.936200
600,15.721900
700,14.375400
800,12.982600
900,11.896300
1000,10.292800


Step,Training Loss
100,76.866400
200,51.778200
300,29.696700
400,21.899500
500,17.936200
600,15.721900
700,14.375400
800,12.982600
900,11.896300
1000,10.292800


Done training.

--- Classification Report (10-class) ---
                     precision    recall  f1-score   support

            DATASET     0.5602    0.8247    0.6672       485
            GENERIC     0.2838    0.6888    0.4019       241
           MATERIAL     0.4410    0.4494    0.4451       158
             METHOD     0.3977    0.5724    0.4693       774
             METRIC     0.9540    0.7249    0.8238      1432
              MODEL     0.9271    0.8467    0.8851      4716
OTHERSCIENTIFICTERM     0.5088    0.6115    0.5555       520
               TASK     0.5785    0.8419    0.6858      2258
         TECHNOLOGY     0.6470    0.9227    0.7607      5151
              TOOLS     0.6209    0.3702    0.4638      3031

          micro avg     0.6754    0.7570    0.7138     18766
          macro avg     0.5919    0.6853    0.6158     18766
       weighted avg     0.7056    0.7570    0.7124     18766


Overall Precision: 0.6754
Overall Recall:    0.7570
Overall F1:        0.7138

--- Co

# roberta

In [ ]:
# =====پیش‌نیازها =====
!pip install TorchCRF pytorch-crf
!pip install --quiet transformers datasets seqeval torch evaluate torchcrf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [ ]:
# ===== کتابخانه‌ها =====
import os, json
import torch
import numpy as np
import pandas as pd
from collections import Counter
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint
from torchcrf import CRF
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import List, Dict, Any
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix


# ===== نام مدل =====
MODEL_NAME = "roberta-base"

torch.backends.cuda.matmul.allow_tf32 = True

if torch.cuda.is_available():
    torch.cuda.empty_cache()

def read_iob_file(file_path):
    sentences, labels = [], []
    tokens, tags = [], []

    with open(file_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                parts = line.split()
                tok, tag = parts[0], parts[-1]
                tokens.append(tok)
                tags.append(tag)

        if tokens:
            sentences.append(tokens)
            labels.append(tags)

    return sentences, labels


# ===== مسیرها =====
train_file = "/content/drive/MyDrive/poroje/data/merged_train_iob_fixed.txt"
test_file  = "/content/drive/MyDrive/poroje/data/merged_test_iob_fixed.txt"

train_sents, train_tags = read_iob_file(train_file)
test_sents, test_tags = read_iob_file(test_file)

unique_labels = sorted({
    t for seq in train_tags + test_tags for t in seq
})

combined_labels = ["O"] + [
    l for l in unique_labels if l != "O"
]

label2id = {l: i for i, l in enumerate(combined_labels)}
id2label = {i: l for l, i in label2id.items()}

flat_tags = [t for seq in train_tags for t in seq]

counts = Counter(flat_tags)
total = sum(counts.values())
num_lbls = len(combined_labels)

class_weights = {
    lbl: total / (num_lbls * counts.get(lbl, 1))
    for lbl in combined_labels
}

weights_tensor = torch.tensor(
    [class_weights[lbl] for lbl in combined_labels],
    dtype=torch.float
)
for cls in [
    "I-TASK", "I-DATASET", "B-TECHNOLOGY", "I-TECHNOLOGY",
    "B-TOOLS", "I-TOOLS", "B-GENERIC", "I-GENERIC",
    "B-MATERIAL", "I-MATERIAL"
]:
    if cls in label2id:
        weights_tensor[label2id[cls]] *= 2.5

train_examples = [
    {"tokens": s, "ner_tags": [label2id[t] for t in ts]}
    for s, ts in zip(train_sents, train_tags)
]

eval_examples = [
    {"tokens": s, "ner_tags": [label2id[t] for t in ts]}
    for s, ts in zip(test_sents, test_tags)
]

datasets = DatasetDict({
    "train": Dataset.from_list(train_examples),
    "eval": Dataset.from_list(eval_examples),
})

print(datasets)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    add_prefix_space=True
)

MAX_LEN = 320


def tokenize_and_align_iob(ex):
    enc = tokenizer(
        ex["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    word_ids = enc.word_ids()
    _ = enc.pop("offset_mapping")

    labels_aln, token_mask = [], []
    prev = None

    for wi in word_ids:

        if wi is None:
            labels_aln.append(label2id["O"])
            token_mask.append(0)

        elif wi != prev:
            labels_aln.append(ex["ner_tags"][wi])
            token_mask.append(1)

        else:
            tag = combined_labels[ex["ner_tags"][wi]]

            if tag.startswith("B-"):
                tag = tag.replace("B-", "I-")

            labels_aln.append(label2id[tag])
            token_mask.append(1)

        prev = wi

    enc["labels"] = labels_aln
    enc["token_mask"] = token_mask

    return enc


tokenized = datasets.map(
    tokenize_and_align_iob,
    batched=False,
    remove_columns=["tokens", "ner_tags"]
)

print(tokenized)

@dataclass
class MyTokenCollator:
    tokenizer: Any
    label_pad_token_id: int

    def __call__(
        self,
        features: List[Dict[str, Any]]
    ) -> Dict[str, torch.Tensor]:

        labels = [f.pop("labels") for f in features]
        masks = [f.pop("token_mask") for f in features]

        batch = self.tokenizer.pad(
            features,
            return_tensors="pt"
        )

        max_len = batch["input_ids"].size(1)

        def pad_to(x, pad_val, L):
            return x + [pad_val] * (L - len(x))

        batch["labels"] = torch.tensor(
            [
                pad_to(l, self.label_pad_token_id, max_len)
                for l in labels
            ],
            dtype=torch.long
        )

        batch["token_mask"] = torch.tensor(
            [
                pad_to(m, 0, max_len)
                for m in masks
            ],
            dtype=torch.bool
        )

        return batch


data_collator = MyTokenCollator(
    tokenizer,
    label_pad_token_id=label2id["O"]
)


# ===== مدل RoBERTa + WordTransformer + Linear + CRF
#        +  Boundary Head =====
class BertCRFForTokenClassification(nn.Module):

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        labels_vocab,
        feature_extraction: bool = False,
        class_weights: torch.Tensor = None,
        focal_alpha: float = 2.0,
        focal_gamma: float = 3.0,
        boundary_beta: float = 0.3
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        if feature_extraction:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.dropout1 = nn.Dropout(0.3)
        self.dropout2 = nn.Dropout(0.3)

        self.word_transformer_layer = TransformerEncoderLayer(
            d_model=self.bert.config.hidden_size,
            nhead=8,
            dim_feedforward=self.bert.config.hidden_size * 4,
            batch_first=True
        )

        self.word_transformer = TransformerEncoder(
            self.word_transformer_layer,
            num_layers=1
        )

        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_labels
        )

        self.crf = CRF(
            num_labels,
            batch_first=True
        )
        self.boundary_classifier = nn.Linear(
            self.bert.config.hidden_size,
            2
        )

        self.class_weights = class_weights
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.boundary_beta = boundary_beta
        self.labels_vocab = labels_vocab
        is_B_list = [
            1 if lbl.startswith("B-") else 0
            for lbl in labels_vocab
        ]

        self.register_buffer(
            "is_B",
            torch.tensor(is_B_list, dtype=torch.long)
        )

        self._apply_bio_constraints()


    def _apply_bio_constraints(self):
        L = self.labels_vocab

        def typ(lbl):
            return lbl.split('-', 1)[1] if lbl != "O" else ""

        with torch.no_grad():
            for i, li in enumerate(L):

                if li.startswith("I-"):
                    self.crf.start_transitions[i] = -1e4

                for j, lj in enumerate(L):

                    if lj.startswith("I-"):
                        ok_prev = (
                            li == f"B-{typ(lj)}"
                        ) or (
                            li == f"I-{typ(lj)}"
                        )

                        if not ok_prev:
                            self.crf.transitions[i, j] = -1e4


    def forward(
        self,
        input_ids,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        token_mask=None
    ):

        x = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        ).last_hidden_state

        x = self.dropout1(x)

        valid_mask = (
            token_mask.bool()
            if token_mask is not None
            else attention_mask.bool()
        )

        x = self.word_transformer(
            x,
            src_key_padding_mask=~valid_mask
        )

        x = self.dropout2(x)

        emissions = self.classifier(x)

        boundary_logits = self.boundary_classifier(x)

        if labels is not None:

            b, s, n = emissions.size()

            em = emissions.reshape(-1, n)[
                valid_mask.reshape(-1)
            ]

            lb = labels.reshape(-1)[
                valid_mask.reshape(-1)
            ]

            ce = F.cross_entropy(
                em,
                lb,
                weight=(
                    self.class_weights.to(em.device)
                    if self.class_weights is not None
                    else None
                ),
                reduction="none"
            )

            pt = torch.exp(-ce)

            focal = (
                self.focal_alpha
                * (1 - pt) ** self.focal_gamma
                * ce
            ).mean()


            crf_loss = -self.crf(
                emissions,
                labels,
                mask=valid_mask,
                reduction="mean"
            )
          
            boundary_targets = self.is_B[labels]

            bl = boundary_logits.reshape(-1, 2)[
                valid_mask.reshape(-1)
            ]

            bt = boundary_targets.reshape(-1)[
                valid_mask.reshape(-1)
            ]

            boundary_loss = F.cross_entropy(
                bl,
                bt,
                reduction="mean"
            )

            total_loss = (
                crf_loss
                + focal
                + self.boundary_beta * boundary_loss
            )

            return {
                "loss": total_loss,
                "logits": emissions,
                "boundary_loss": boundary_loss.detach()
            }

        best_paths = self.crf.decode(
            emissions,
            mask=valid_mask
        )

        max_len = emissions.size(1)

        padded = [
            seq + [0] * (max_len - len(seq))
            for seq in best_paths
        ]

        return torch.tensor(
            padded,
            device=emissions.device
        )

EPOCHS = 30

CHECKPOINT_DIR = "./temp"

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)


model = BertCRFForTokenClassification(
    MODEL_NAME,
    len(combined_labels),
    labels_vocab=combined_labels,
    feature_extraction=False,
    class_weights=weights_tensor,
    focal_alpha=2.0,
    focal_gamma=3.0,
    boundary_beta=0.3
)


training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.1,
    warmup_ratio=0.10,

    logging_dir=os.path.join(CHECKPOINT_DIR, "logs"),
    logging_steps=100,

    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    load_best_model_at_end=False,

    bf16=True,
    tf32=True,
    optim="adamw_torch_fused",

    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    do_train=True,
    do_eval=True,
    report_to=[]
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

last_ckpt = get_last_checkpoint(CHECKPOINT_DIR)

trainer.train(
    resume_from_checkpoint=(
        last_ckpt if last_ckpt is not None else None
    )
)

eval_results = trainer.evaluate()

print("\n=== Eval (HF; loss-based) ===")
print(eval_results)

eval_loader = DataLoader(
    tokenized["eval"],
    batch_size=16,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=2
)

model.eval()

device = next(model.parameters()).device

all_true, all_pred = [], []


with torch.no_grad():

    for batch in eval_loader:

        labels = batch.pop("labels")
        tmask = batch["token_mask"]

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        decoded_ids = model(**batch).cpu()

        for true_ids, pred_ids, m in zip(
            labels,
            decoded_ids,
            tmask
        ):

            seq_true, seq_pred = [], []

            for t_id, p_id, mk in zip(
                true_ids,
                pred_ids,
                m
            ):

                if mk:
                    seq_true.append(
                        id2label[int(t_id)]
                    )

                    seq_pred.append(
                        id2label[int(p_id)]
                    )

            all_true.append(seq_true)
            all_pred.append(seq_pred)


# ===== Classification Report =====
print("\n--- Classification Report")

print(
    classification_report(
        all_true,
        all_pred,
        digits=4
    )
)

print(
    f"\nOverall Precision: {precision_score(all_true, all_pred):.4f}"
)

print(
    f"Overall Recall:    {recall_score(all_true, all_pred):.4f}"
)

print(
    f"Overall F1:        {f1_score(all_true, all_pred):.4f}"
)


# ===== ماتریس درهم‌ریختگی =====
flat_true = [
    lbl
    for seq in all_true
    for lbl in seq
]

flat_pred = [
    lbl
    for seq in all_pred
    for lbl in seq
]

cm = confusion_matrix(
    flat_true,
    flat_pred,
    labels=combined_labels
)

df_cm = pd.DataFrame(
    cm,
    index=combined_labels,
    columns=combined_labels
)

print("\n--- Confusion Matrix ---")
print(df_cm)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 57263
    })
    eval: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14524
    })
})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

Step,Training Loss
100,63.100400
200,50.598500
300,35.318600
400,32.644900
500,25.191700
600,25.825400
700,23.786700
800,25.358700
900,19.439900
1000,15.862900


You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenize


=== Eval (HF; loss-based) ===
{'eval_loss': 15.247269630432129, 'eval_runtime': 55.0764, 'eval_samples_per_second': 263.706, 'eval_steps_per_second': 8.243, 'epoch': 30.0}


You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



--- Classification Report
                     precision    recall  f1-score   support

            DATASET     0.6716    0.8392    0.7461       485
            GENERIC     0.4190    0.6971    0.5234       241
           MATERIAL     0.4970    0.5190    0.5077       158
             METHOD     0.4515    0.6000    0.5152       775
             METRIC     0.9351    0.6236    0.7482      1432
              MODEL     0.9170    0.6983    0.7928      4716
OTHERSCIENTIFICTERM     0.5165    0.6327    0.5687       520
               TASK     0.5787    0.8336    0.6831      2259
         TECHNOLOGY     0.8039    0.8435    0.8232      5151
              TOOLS     0.5121    0.6447    0.5708      3031

          micro avg     0.6958    0.7363    0.7155     18768
          macro avg     0.6302    0.6932    0.6479     18768
       weighted avg     0.7346    0.7363    0.7240     18768


Overall Precision: 0.6958
Overall Recall:    0.7363
Overall F1:        0.7155

--- Confusion Matrix ---
           

# bert-base

In [ ]:
# =====پیش‌نیازها =====
!pip install TorchCRF pytorch-crf
!pip install --quiet transformers datasets seqeval torch evaluate torchcrf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


In [ ]:
# ===== وارد کردن کتابخانه‌ها =====
import os, json
import torch
import numpy as np
import pandas as pd
from collections import Counter
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint
from torchcrf import CRF
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import List, Dict, Any
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix

# ===== 0) نام مدل =====
MODEL_NAME = "bert-base-cased"

torch.backends.cuda.matmul.allow_tf32 = True
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def read_iob_file(file_path):
    sentences, labels = [], []
    tokens, tags = [], []

    with open(file_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                parts = line.split()
                tok, tag = parts[0], parts[-1]
                tokens.append(tok)
                tags.append(tag)

        if tokens:
            sentences.append(tokens)
            labels.append(tags)

    return sentences, labels


# مسیرها
train_file = "/content/drive/MyDrive/poroje/data/merged_train_iob_fixed.txt"
test_file  = "/content/drive/MyDrive/poroje/data/merged_test_iob_fixed.txt"

train_sents, train_tags = read_iob_file(train_file)
test_sents, test_tags = read_iob_file(test_file)

unique_labels = sorted({
    t for seq in train_tags + test_tags for t in seq
})

combined_labels = ["O"] + [
    l for l in unique_labels if l != "O"
]

label2id = {l: i for i, l in enumerate(combined_labels)}
id2label = {i: l for l, i in label2id.items()}

flat_tags = [t for seq in train_tags for t in seq]
counts = Counter(flat_tags)
total = sum(counts.values())
num_lbls = len(combined_labels)

class_weights = {
    lbl: total / (num_lbls * counts.get(lbl, 1))
    for lbl in combined_labels
}

weights_tensor = torch.tensor(
    [class_weights[lbl] for lbl in combined_labels],
    dtype=torch.float
)

for cls in [
    "I-TASK", "I-DATASET",
    "B-TECHNOLOGY", "I-TECHNOLOGY",
    "B-TOOLS", "I-TOOLS",
    "B-GENERIC", "I-GENERIC",
    "B-MATERIAL", "I-MATERIAL"
]:
    if cls in label2id:
        weights_tensor[label2id[cls]] *= 2.5

train_examples = [
    {
        "tokens": s,
        "ner_tags": [label2id[t] for t in ts]
    }
    for s, ts in zip(train_sents, train_tags)
]

eval_examples = [
    {
        "tokens": s,
        "ner_tags": [label2id[t] for t in ts]
    }
    for s, ts in zip(test_sents, test_tags)
]

datasets = DatasetDict({
    "train": Dataset.from_list(train_examples),
    "eval": Dataset.from_list(eval_examples),
})

print(datasets)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

MAX_LEN = 320


def tokenize_and_align_iob(ex):
    enc = tokenizer(
        ex["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    word_ids = enc.word_ids()
    _ = enc.pop("offset_mapping")

    labels_aln, token_mask = [], []
    prev = None

    for wi in word_ids:
        if wi is None:
            labels_aln.append(label2id["O"])
            token_mask.append(0)

        elif wi != prev:
            labels_aln.append(ex["ner_tags"][wi])
            token_mask.append(1)

        else:
            tag = combined_labels[ex["ner_tags"][wi]]

            if tag.startswith("B-"):
                tag = tag.replace("B-", "I-")

            labels_aln.append(label2id[tag])
            token_mask.append(1)

        prev = wi

    enc["labels"] = labels_aln
    enc["token_mask"] = token_mask

    return enc


tokenized = datasets.map(
    tokenize_and_align_iob,
    batched=False,
    remove_columns=["tokens", "ner_tags"]
)

print(tokenized)

@dataclass
class MyTokenCollator:
    tokenizer: Any
    label_pad_token_id: int

    def __call__(
        self,
        features: List[Dict[str, Any]]
    ) -> Dict[str, torch.Tensor]:

        labels = [f.pop("labels") for f in features]
        masks = [f.pop("token_mask") for f in features]

        batch = self.tokenizer.pad(
            features,
            return_tensors="pt"
        )

        max_len = batch["input_ids"].size(1)

        def pad_to(x, pad_val, L):
            return x + [pad_val] * (L - len(x))

        batch["labels"] = torch.tensor(
            [
                pad_to(l, self.label_pad_token_id, max_len)
                for l in labels
            ],
            dtype=torch.long
        )

        batch["token_mask"] = torch.tensor(
            [
                pad_to(m, 0, max_len)
                for m in masks
            ],
            dtype=torch.bool
        )

        return batch


data_collator = MyTokenCollator(
    tokenizer,
    label_pad_token_id=label2id["O"]
)


# ===== مدل BERT + Transformer + CRF + Auxiliary Boundary Head =====
class BertCRFForTokenClassification(nn.Module):

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        labels_vocab,
        feature_extraction: bool = False,
        class_weights: torch.Tensor = None,
        focal_alpha: float = 2.0,
        focal_gamma: float = 3.0,
        boundary_beta: float = 0.3
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        if feature_extraction:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.dropout1 = nn.Dropout(0.3)
        self.dropout2 = nn.Dropout(0.3)

        self.word_transformer_layer = TransformerEncoderLayer(
            d_model=self.bert.config.hidden_size,
            nhead=8,
            dim_feedforward=self.bert.config.hidden_size * 4,
            batch_first=True
        )

        self.word_transformer = TransformerEncoder(
            self.word_transformer_layer,
            num_layers=1
        )

        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_labels
        )

        self.crf = CRF(
            num_labels,
            batch_first=True
        )

        self.class_weights = class_weights
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.labels_vocab = labels_vocab

        self.boundary_classifier = nn.Linear(
            self.bert.config.hidden_size,
            2
        )

        self.boundary_beta = boundary_beta

        is_B = torch.tensor(
            [
                1 if lbl.startswith("B-") else 0
                for lbl in self.labels_vocab
            ],
            dtype=torch.long
        )

        self.register_buffer("is_B", is_B)

        self._apply_bio_constraints()

    def _apply_bio_constraints(self):
        L = self.labels_vocab

        def typ(lbl):
            return lbl.split('-', 1)[1] if lbl != "O" else ""

        with torch.no_grad():
            for i, li in enumerate(L):

                if li.startswith("I-"):
                    self.crf.start_transitions[i] = -1e4

                for j, lj in enumerate(L):

                    if lj.startswith("I-"):
                        ok_prev = (
                            li == f"B-{typ(lj)}"
                            or li == f"I-{typ(lj)}"
                        )

                        if not ok_prev:
                            self.crf.transitions[i, j] = -1e4

    def forward(
        self,
        input_ids,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        token_mask=None
    ):

       
        x = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        ).last_hidden_state

        x = self.dropout1(x)

        valid_mask = (
            token_mask.bool()
            if token_mask is not None
            else attention_mask.bool()
        )

        
        x = self.word_transformer(
            x,
            src_key_padding_mask=~valid_mask
        )

        x = self.dropout2(x)

      
        emissions = self.classifier(x)

        boundary_logits = self.boundary_classifier(x)

        if labels is not None:

            
            b, s, n = emissions.size()

            em = emissions.reshape(-1, n)[
                valid_mask.reshape(-1)
            ]

            lb = labels.reshape(-1)[
                valid_mask.reshape(-1)
            ]

            ce = F.cross_entropy(
                em,
                lb,
                weight=(
                    self.class_weights.to(em.device)
                    if self.class_weights is not None
                    else None
                ),
                reduction="none"
            )

            pt = torch.exp(-ce)

            focal = (
                self.focal_alpha
                * (1 - pt) ** self.focal_gamma
                * ce
            ).mean()

            crf_loss = -self.crf(
                emissions,
                labels,
                mask=valid_mask,
                reduction="mean"
            )

            boundary_targets = self.is_B[labels]

            valid = valid_mask.bool()

            bl = boundary_logits[valid]
            bt = boundary_targets[valid]

            if bt.numel() > 0:
                boundary_loss = F.cross_entropy(
                    bl,
                    bt
                )
            else:
                boundary_loss = emissions.sum() * 0.0

            # ===== Total Loss =====
            total_loss = (
                crf_loss
                + focal
                + self.boundary_beta * boundary_loss
            )

            return {
                "loss": total_loss,
                "logits": emissions
            }

        best_paths = self.crf.decode(
            emissions,
            mask=valid_mask
        )

        max_len = emissions.size(1)

        padded = [
            seq + [0] * (max_len - len(seq))
            for seq in best_paths
        ]

        return torch.tensor(
            padded,
            device=emissions.device
        )


# ===== 7) TrainingArguments=====
EPOCHS = 30

CHECKPOINT_DIR = "./temp"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model = BertCRFForTokenClassification(
    MODEL_NAME,
    len(combined_labels),
    labels_vocab=combined_labels,
    feature_extraction=False,
    class_weights=weights_tensor,
    focal_alpha=2.0,
    focal_gamma=3.0,
    boundary_beta=0.3
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.1,
    warmup_ratio=0.10,
    logging_dir=os.path.join(CHECKPOINT_DIR, "logs"),
    logging_steps=100,

    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    load_best_model_at_end=False,

    bf16=True,
    tf32=True,
    optim="adamw_torch_fused",
    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    do_train=True,
    do_eval=True,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

last_ckpt = get_last_checkpoint(CHECKPOINT_DIR)

trainer.train(
    resume_from_checkpoint=(
        last_ckpt if last_ckpt is not None else None
    )
)

eval_results = trainer.evaluate()

print("\n=== Eval (HF; loss-based) ===")
print(eval_results)

eval_loader = DataLoader(
    tokenized["eval"],
    batch_size=16,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=2
)

model.eval()

device = next(model.parameters()).device

all_true, all_pred = [], []

with torch.no_grad():

    for batch in eval_loader:

        labels = batch.pop("labels")
        tmask = batch["token_mask"]

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        decoded_ids = model(**batch).cpu()

        for true_ids, pred_ids, m in zip(
            labels,
            decoded_ids,
            tmask
        ):

            seq_true, seq_pred = [], []

            for t_id, p_id, mk in zip(
                true_ids,
                pred_ids,
                m
            ):

                if mk:
                    seq_true.append(
                        id2label[int(t_id)]
                    )

                    seq_pred.append(
                        id2label[int(p_id)]
                    )

            all_true.append(seq_true)
            all_pred.append(seq_pred)


print("\n--- Classification Report")
print(
    classification_report(
        all_true,
        all_pred,
        digits=4
    )
)

print(
    f"\nOverall Precision: {precision_score(all_true, all_pred):.4f}"
)

print(
    f"Overall Recall:    {recall_score(all_true, all_pred):.4f}"
)

print(
    f"Overall F1:        {f1_score(all_true, all_pred):.4f}"
)


# ===== ماتریس درهم‌ریختگی =====
flat_true = [
    lbl for seq in all_true for lbl in seq
]

flat_pred = [
    lbl for seq in all_pred for lbl in seq
]

cm = confusion_matrix(
    flat_true,
    flat_pred,
    labels=combined_labels
)

df_cm = pd.DataFrame(
    cm,
    index=combined_labels,
    columns=combined_labels
)

print("\n--- Confusion Matrix ---")
print(df_cm)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 57263
    })
    eval: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14524
    })
})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

...

Step,Training Loss
100,79.392200
200,57.444200
300,37.973300
400,35.406300
500,27.882300
600,29.355400
700,25.464100
800,27.008800
900,22.403600
1000,17.499700


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `_


=== Eval (HF; loss-based) ===
{'eval_loss': 19.680675506591797, 'eval_runtime': 54.5047, 'eval_samples_per_second': 266.472, 'eval_steps_per_second': 8.33, 'epoch': 30.0}


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



--- Classification Report
                     precision    recall  f1-score   support

            DATASET     0.5609    0.8165    0.6650       485
            GENERIC     0.2870    0.6432    0.3969       241
           MATERIAL     0.2229    0.4430    0.2966       158
             METHOD     0.2563    0.5652    0.3527       775
             METRIC     0.9336    0.6774    0.7851      1432
              MODEL     0.9217    0.7886    0.8500      4716
OTHERSCIENTIFICTERM     0.3572    0.6135    0.4515       520
               TASK     0.6596    0.8132    0.7284      2259
         TECHNOLOGY     0.8324    0.9018    0.8657      5151
              TOOLS     0.4515    0.3042    0.3635      3031

          micro avg     0.6858    0.7178    0.7014     18768
          macro avg     0.5483    0.6566    0.5755     18768
       weighted avg     0.7242    0.7178    0.7093     18768


Overall Precision: 0.6858
Overall Recall:    0.7178
Overall F1:        0.7014

--- Confusion Matrix ---
           